# WAFER-GUARD · Fab-7 Excursion Detector — **repaired** pipeline

This is the audited and corrected version of the original notebook (`bnypi.ipynb`).
Every change is tagged in the code with `# FIX[n]` and explained in `AUDIT.md`.

| # | Silent defect in the original | Repair in this notebook |
|---|---|---|
| 1 | **Data leakage.** The imputer and scaler were fit on all rows, SMOTE ran *before* the split, and the split was random, so the future and the test set leaked into training. The test set was also 50 % synthetic and balanced. | Chronological train / val / test split. All statistics are fit on **train only**. No SMOTE: class imbalance is handled with `pos_weight`. The test set keeps its natural prevalence. |
| 2 | **Corrupted loss gradients.** `reconstructed` is `[B,F]` and `batch_x` is `[B,1,F]`, so they broadcast silently to `[B,B,F]` and the decoder learns the batch mean. | Tensors are kept at `[B,F]`. A loss function asserts the shapes and masks missing or placeholder cells. |
| 3 | **Invariant blindness.** The legacy PS-L sentinel (`sensor_0 == 0.0`) and the firmware codes in `sensor_1` were scaled and imputed as continuous values. | A rule-based legacy flag and one-hot error codes. The sentinel and code slots are excluded from scaling, imputation statistics and the reconstruction loss. |

The notebook has two stages:
* **Stage A (sections 1–11).** The original model and hyperparameters with only the bugs fixed. This isolates the effect of the repairs.
* **Stage B (sections 12–14).** A tuning round that is pre-registered (the grid is fixed before any result is seen) and leakage-safe. Configurations are chosen by walk-forward cross-validation inside the train+val history only. The chosen model is then scored **once** on the untouched test window, and the release gate uses that result.

## 0 · Environment

In [1]:
# Dependencies: see requirements.txt  ->  pip install -r requirements.txt
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve,
                             classification_report)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=RuntimeWarning)   # nan-reductions on all-NaN columns

SEED = 1337
def seed_all(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_all(SEED)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")          # tiny tabular problem; CPU keeps runs bit-reproducible

HP = dict(batch_size=64, max_epochs=60, patience=10, lr=1e-3, weight_decay=1e-4,
          hidden=256, latent=32, lambda_clf=1.0,
          split=(0.60, 0.20, 0.20),          # chronological train / val / test
          max_missing=0.50,                  # drop channels missing in >50 % of TRAIN rows
          indicator_min_missing=0.01,        # add a missing-flag for channels missing >=1 % in TRAIN
          z_clip=10.0)                       # robust to drifted/outlier telemetry at inference
NATURAL_PREVALENCE = 0.0664                  # full-dataset defect rate (104 / 1567)
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.12.1+cpu | device=cpu


## 1 · Data
Rows are sorted by `timestamp`. The fab drifts over time, so time order matters for every later step.

In [2]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
assert df["timestamp"].is_monotonic_increasing

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
y_all = (df["label"].to_numpy() == 1).astype(np.int64)            # 1 = defective

print(f"rows {len(df)} | sensors {len(SENSOR_COLS)} | defect rate {y_all.mean():.2%} "
      f"({y_all.sum()} fail / {(1 - y_all).sum()} pass)")
print(f"span {df.timestamp.min()} -> {df.timestamp.max()}")
print("defect rate per chronological decile:",
      [f"{c.mean():.1%}" for c in np.array_split(y_all, 10)])

rows 1567 | sensors 590 | defect rate 6.64% (104 fail / 1463 pass)
span 2008-07-19 11:55:00 -> 2008-10-17 06:07:00
defect rate per chronological decile: ['12.7%', '15.3%', '8.9%', '4.5%', '1.3%', '5.7%', '1.3%', '6.4%', '8.3%', '1.9%']


## 2 · Chronological split  `# FIX[1]`
Oldest 60 % → **train**, next 20 % → **validation** (early stopping and threshold), newest 20 % → **test**.
The test window is never resampled. Its defect rate is whatever the fab actually produced in that period.

In [3]:
n = len(df)
i1 = int(HP["split"][0] * n); i2 = int((HP["split"][0] + HP["split"][1]) * n)
idx_tr, idx_va, idx_te = np.arange(0, i1), np.arange(i1, i2), np.arange(i2, n)

# causality: every training timestamp strictly precedes every val timestamp, and so on
assert df.timestamp[idx_tr].max() < df.timestamp[idx_va].min()
assert df.timestamp[idx_va].max() < df.timestamp[idx_te].min()

for name, ix in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{name:5s} n={len(ix):4d}  defects={y_all[ix].sum():3d} ({y_all[ix].mean():.2%})  "
          f"{df.timestamp[ix].min()} -> {df.timestamp[ix].max()}")

train n= 940  defects= 76 (8.09%)  2008-07-19 11:55:00 -> 2008-09-20 05:34:00
val   n= 313  defects= 11 (3.51%)  2008-09-20 06:08:00 -> 2008-10-02 19:25:00
test  n= 314  defects= 17 (5.41%)  2008-10-02 20:54:00 -> 2008-10-17 06:07:00


## 3 · Legacy Probe Station PS-L tool-matching  `# FIX[3]`
* `sensor_0 == 0.0` is a **sentinel** meaning "routed through PS-L". It is not a measurement.
* On PS-L wafers `sensor_1` is a **categorical firmware error code** (100 / 200 / 300), not a reading.

These rules apply **row by row** and have no fitted state, so they cannot leak information across time.
A code we have never seen is routed to `s1_code_unknown`. It is not silently treated as a number.

In [4]:
LEGACY_SENTINEL = 0.0
LEGACY_CODES = (100.0, 200.0, 300.0)

def legacy_view(frame: pd.DataFrame):
    # Returns (is_legacy[N], code_onehot[N,4], na_mask[N,F]) for one frame. Stateless, row-wise.
    s0 = frame["sensor_0"].to_numpy(np.float64)
    s1 = frame["sensor_1"].to_numpy(np.float64)
    is_legacy = (s0 == LEGACY_SENTINEL)
    known = np.stack([s1 == c for c in LEGACY_CODES], axis=1)
    # a legacy row whose sensor_1 is not a known code -> 'unknown code' bucket
    unknown = is_legacy & ~known.any(1)
    # defensive: a known code value on a non-legacy row is still a code, never a reading
    code_slot = is_legacy | known.any(1)
    onehot = np.concatenate([known & code_slot[:, None], unknown[:, None]], 1).astype(np.float32)
    na = np.zeros((len(frame), len(SENSOR_COLS)), dtype=bool)   # 'not applicable' slots
    na[is_legacy, SENSOR_COLS.index("sensor_0")] = True
    na[code_slot, SENSOR_COLS.index("sensor_1")] = True
    return is_legacy, onehot, na

is_leg, codes, _ = legacy_view(df)
print(f"legacy PS-L wafers: {is_leg.sum()} ({is_leg.mean():.1%})")
print("code counts (100/200/300/unknown):", codes.sum(0).astype(int).tolist())
print("defect rate legacy vs current-gen:",
      f"{y_all[is_leg].mean():.2%} vs {y_all[~is_leg].mean():.2%}")

legacy PS-L wafers: 146 (9.3%)
code counts (100/200/300/unknown): [61, 35, 50, 0]
defect rate legacy vs current-gen: 0.00% vs 7.32%


## 4 · Causal preprocessor  `# FIX[1]` `# FIX[3]`
`fit()` sees **training rows only**. `transform()` is then applied unchanged to validation, test and any future production wafer.

* Legacy sentinel and code slots become *not-applicable*. They are excluded from every mean, std and median.
* Channels that are constant, or missing in more than 50 % of rows, **in train** are dropped.
* Remaining channels are z-scored with train statistics and clipped to ±10σ.
* Genuinely missing cells get the train median plus a missing-indicator, for channels with ≥1 % train missingness.
* Not-applicable legacy slots get a fixed 0 placeholder. It is never used as a target (`recon_mask = 0`). The model learns its meaning from `is_legacy` and the one-hot codes.

In [5]:
class CausalPreprocessor:
    def fit(self, frame: pd.DataFrame):
        X = frame[SENSOR_COLS].to_numpy(np.float64).copy()
        _, _, na = legacy_view(frame)
        X[na] = np.nan                                   # sentinel / codes never enter statistics
        miss = np.isnan(X).mean(0)
        std = np.nanstd(X, 0)
        self.keep_ = np.where((miss <= HP["max_missing"]) & (np.nan_to_num(std) > 0))[0]
        Xk = X[:, self.keep_]
        self.mean_, self.std_ = np.nanmean(Xk, 0), np.nanstd(Xk, 0)
        self.median_z_ = (np.nanmedian(Xk, 0) - self.mean_) / self.std_
        genuine_miss = (np.isnan(frame[SENSOR_COLS].to_numpy(np.float64)) & ~na)[:, self.keep_].mean(0)
        self.ind_ = np.where(genuine_miss >= HP["indicator_min_missing"])[0]
        self.feature_names_ = ([SENSOR_COLS[k] for k in self.keep_]
                               + [f"miss_{SENSOR_COLS[self.keep_[j]]}" for j in self.ind_]
                               + ["is_legacy", "s1_code_100", "s1_code_200", "s1_code_300", "s1_code_unknown"])
        self.n_cont_ = len(self.keep_)
        return self

    def transform(self, frame: pd.DataFrame):
        raw = frame[SENSOR_COLS].to_numpy(np.float64)
        is_legacy, onehot, na = legacy_view(frame)
        Xk = raw[:, self.keep_]
        na_k = na[:, self.keep_]
        miss_k = np.isnan(Xk) & ~na_k                    # genuinely missing readings
        Z = np.clip((Xk - self.mean_) / self.std_, -HP["z_clip"], HP["z_clip"])
        Z = np.where(miss_k, self.median_z_, Z)          # train-median imputation
        Z = np.where(na_k, 0.0, Z)                       # placeholder for not-applicable slots
        feats = np.concatenate([Z, miss_k[:, self.ind_].astype(np.float64),
                                is_legacy[:, None].astype(np.float64), onehot], 1).astype(np.float32)
        # reconstruction targets: only genuinely observed readings + the categorical / indicator block
        mask = np.ones_like(feats, dtype=np.float32)
        mask[:, :self.n_cont_] = (~miss_k & ~na_k).astype(np.float32)
        assert np.isfinite(feats).all()
        return feats, mask

prep = CausalPreprocessor().fit(df.iloc[idx_tr])
(X_tr, M_tr), (X_va, M_va), (X_te, M_te) = (prep.transform(df.iloc[ix]) for ix in (idx_tr, idx_va, idx_te))
y_tr, y_va, y_te = y_all[idx_tr], y_all[idx_va], y_all[idx_te]
F = X_tr.shape[1]
print(f"features: {F} = {prep.n_cont_} continuous + {len(prep.ind_)} missing-flags + 5 legacy/categorical")
j1 = list(prep.keep_).index(SENSOR_COLS.index("sensor_1"))
print(f"sensor_1 train stats used for scaling: mean={prep.mean_[j1]:.2f} std={prep.std_[j1]:.2f} "
      f"(genuine readings only; codes excluded)")
print(f"X_tr {X_tr.shape} | X_va {X_va.shape} | X_te {X_te.shape}")

features: 487 = 444 continuous + 38 missing-flags + 5 legacy/categorical
sensor_1 train stats used for scaling: mean=2494.87 std=85.66 (genuine readings only; codes excluded)
X_tr (940, 487) | X_va (313, 487) | X_te (314, 487)


## 5 · Tensors & loaders  `# FIX[2]`
Features stay **`[N, F]`**. The original added a channel axis (`unsqueeze(1)`), which caused the broadcasting bug.

In [6]:
def make_loader(X, M, y, shuffle, seed=SEED):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(M), torch.from_numpy(y).float())
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=HP["batch_size"], shuffle=shuffle,
                      generator=g if shuffle else None, drop_last=False)

xb, mb, yb = next(iter(make_loader(X_tr, M_tr, y_tr, True)))
print("batch shapes  x", tuple(xb.shape), "| mask", tuple(mb.shape), "| y", tuple(yb.shape))

batch shapes  x (64, 487) | mask (64, 487) | y (64,)


## 6 · Model — supervised autoencoder (same architecture as the original; strict input rank)

In [7]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int, p_drop: float = 0.10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        # FIX[2]: no silent flatten; reject anything that is not [B, F]
        assert x.dim() == 2, f"expected [B, F], got {tuple(x.shape)}"
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

## 7 · Shape-checked loss  `# FIX[2]`
* Reconstruction: masked MSE over **observed cells only**, using an element-wise `[B,F]` comparison.
* Classification: `BCEWithLogits` on `[B]` logits against `[B]` labels. `pos_weight` = train negatives / positives replaces SMOTE.

In [8]:
def sae_loss(recon, x, mask, logits, y, pos_weight, lam):
    B = x.shape[0]
    assert recon.shape == x.shape == mask.shape and x.dim() == 2, (recon.shape, x.shape, mask.shape)
    assert logits.shape == y.shape == (B,), (logits.shape, y.shape)
    recon_loss = ((recon - x) ** 2 * mask).sum() / mask.sum().clamp(min=1.0)
    clf_loss = Fnn.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
    return recon_loss + lam * clf_loss, recon_loss, clf_loss

# regression test: the original [B,1,F] input must now fail loudly instead of broadcasting
_m = WaferSAE(F, 16, 4)
try:
    _m(torch.zeros(4, 1, F)); raise RuntimeError("rank check did not fire")
except AssertionError as e:
    print("OK - rank-3 input rejected:", e)
try:
    sae_loss(torch.zeros(4, F), torch.zeros(4, 1, F), torch.ones(4, 1, F), torch.zeros(4), torch.zeros(4),
             torch.tensor(1.0), 1.0); raise RuntimeError("shape check did not fire")
except AssertionError:
    print("OK - mismatched loss shapes rejected")

OK - rank-3 input rejected: expected [B, F], got (4, 1, 487)
OK - mismatched loss shapes rejected


## 8 · Training (early stopping on validation average precision)

In [9]:
@torch.no_grad()
def predict_proba(model, X):
    model.eval()
    return torch.sigmoid(model(torch.from_numpy(X).to(DEVICE))[1]).cpu().numpy()

def train_sae(X_tr, M_tr, y_tr, X_va, y_va, seed=SEED, verbose=True):
    seed_all(seed)
    model = WaferSAE(X_tr.shape[1], HP["hidden"], HP["latent"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["max_epochs"])
    pos_weight = torch.tensor((1 - y_tr).sum() / max(y_tr.sum(), 1), dtype=torch.float32)
    loader = make_loader(X_tr, M_tr, y_tr, shuffle=True, seed=seed)
    best = dict(ap=-1.0, epoch=0, state=None); hist = []
    for epoch in range(1, HP["max_epochs"] + 1):
        model.train(); tot = rec = clf = 0.0
        for xb, mb, yb in loader:
            r, logit, _ = model(xb.to(DEVICE))
            loss, lr_, lc_ = sae_loss(r, xb, mb, logit, yb, pos_weight, HP["lambda_clf"])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
            tot += loss.item() * len(xb); rec += lr_.item() * len(xb); clf += lc_.item() * len(xb)
        sched.step()
        va_ap = average_precision_score(y_va, predict_proba(model, X_va))
        hist.append(dict(epoch=epoch, total=tot / len(X_tr), recon=rec / len(X_tr), clf=clf / len(X_tr), val_ap=va_ap))
        if va_ap > best["ap"]:
            best = dict(ap=va_ap, epoch=epoch, state={k: v.clone() for k, v in model.state_dict().items()})
        if verbose and (epoch == 1 or epoch % 5 == 0):
            h = hist[-1]
            print(f"epoch {epoch:02d} | total {h['total']:.4f} | recon {h['recon']:.4f} | "
                  f"clf {h['clf']:.4f} | val AP {va_ap:.4f}")
        if epoch - best["epoch"] >= HP["patience"]:
            break
    model.load_state_dict(best["state"])
    if verbose:
        print(f"best epoch {best['epoch']} (val AP {best['ap']:.4f}); pos_weight={pos_weight.item():.2f}")
    return model, pd.DataFrame(hist)

model, history = train_sae(X_tr, M_tr, y_tr, X_va, y_va)

epoch 01 | total 2.0312 | recon 0.7943 | clf 1.2369 | val AP 0.0363


epoch 05 | total 0.7493 | recon 0.7087 | clf 0.0406 | val AP 0.0376


epoch 10 | total 0.6059 | recon 0.6037 | clf 0.0022 | val AP 0.0337


best epoch 3 (val AP 0.0398); pos_weight=11.37


## 9 · Decision threshold (validation window only)
The original used a fixed 0.5 threshold, which is meaningless after training on a 50/50 synthetic distribution.
Here the threshold that maximises F1 on the **validation** window is frozen before the test set is touched.

In [10]:
def pick_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[np.argmax(f1)])

p_va = predict_proba(model, X_va)
THRESHOLD = pick_threshold(y_va, p_va)
print(f"validation: AP {average_precision_score(y_va, p_va):.4f} | ROC-AUC {roc_auc_score(y_va, p_va):.4f} "
      f"| F1@thr {f1_score(y_va, p_va >= THRESHOLD):.4f} | threshold {THRESHOLD:.4f}")

validation: AP 0.0398 | ROC-AUC 0.5536 | F1@thr 0.0933 | threshold 0.0194


## 10 · Final evaluation — untouched test window at natural prevalence
* **Primary metric: PR-AUC (average precision).** At ~5–7 % prevalence ROC-AUC flatters a model. A random scorer's AP equals the prevalence, so the reference point is easy to state.
* F1 / precision / recall are reported at the frozen validation threshold, with 95 % bootstrap CIs (2 000 resamples).
* The test window's measured prevalence differs from the dataset-wide 6.64 % because the fab drifts. So precision and F1 are **also** re-expressed at π = 6.64 %, using the measured TPR/FPR (Bayes' rule, no resampling).

In [11]:
def evaluate(y, p, thr, n_boot=2000, seed=SEED):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr, fpr = tp / max(tp + fn, 1), fp / max(fp + tn, 1)
    pi = NATURAL_PREVALENCE
    prec_pi = tpr * pi / max(tpr * pi + fpr * (1 - pi), 1e-12)
    out = dict(n=len(y), defects=int(y.sum()), prevalence=y.mean(),
               ap=average_precision_score(y, p), roc_auc=roc_auc_score(y, p),
               f1=f1_score(y, pred, zero_division=0), precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred), tp=tp, fp=fp, fn=fn, tn=tn,
               precision_at_6_64=prec_pi,
               f1_at_6_64=2 * prec_pi * tpr / max(prec_pi + tpr, 1e-12))
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):
        b = rng.integers(0, len(y), len(y))
        if y[b].sum() in (0, len(b)):
            continue
        boots.append((average_precision_score(y[b], p[b]), roc_auc_score(y[b], p[b]),
                      f1_score(y[b], p[b] >= thr, zero_division=0)))
    lo, hi = np.percentile(np.array(boots), [2.5, 97.5], axis=0)
    out.update(ap_ci=(lo[0], hi[0]), roc_auc_ci=(lo[1], hi[1]), f1_ci=(lo[2], hi[2]))
    return out

def show(r, title):
    print(f"--- {title} ---")
    print(f"n={r['n']}  defects={r['defects']}  prevalence={r['prevalence']:.2%}")
    print(f"PR-AUC (AP) {r['ap']:.4f}  95% CI [{r['ap_ci'][0]:.3f}, {r['ap_ci'][1]:.3f}]   "
          f"(random = {r['prevalence']:.4f})")
    print(f"ROC-AUC     {r['roc_auc']:.4f}  95% CI [{r['roc_auc_ci'][0]:.3f}, {r['roc_auc_ci'][1]:.3f}]")
    print(f"F1          {r['f1']:.4f}  95% CI [{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}]   "
          f"precision {r['precision']:.4f}  recall {r['recall']:.4f}")
    print(f"confusion   TN={r['tn']}  FP={r['fp']}  FN={r['fn']}  TP={r['tp']}")
    print(f"at natural prevalence 6.64%: precision {r['precision_at_6_64']:.4f}  F1 {r['f1_at_6_64']:.4f}")

p_te = predict_proba(model, X_te)
res_sae = evaluate(y_te, p_te, THRESHOLD)
show(res_sae, "WaferSAE - chronological test window")
print(classification_report(y_te, (p_te >= THRESHOLD).astype(int), target_names=["pass", "defect"], digits=4))

--- WaferSAE - chronological test window ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.0696  95% CI [0.039, 0.155]   (random = 0.0541)
ROC-AUC     0.5389  95% CI [0.392, 0.687]
F1          0.1364  95% CI [0.047, 0.239]   precision 0.0845  recall 0.3529
confusion   TN=232  FP=65  FN=11  TP=6
at natural prevalence 6.64%: precision 0.1029  F1 0.1593
              precision    recall  f1-score   support

        pass     0.9547    0.7811    0.8593       297
      defect     0.0845    0.3529    0.1364        17

    accuracy                         0.7580       314
   macro avg     0.5196    0.5670    0.4978       314
weighted avg     0.9076    0.7580    0.8201       314



## 11 · Robustness checks
1. **Seed variance.** Retrain with 5 seeds. Every seed uses the same causal split and its own validation-chosen threshold.
2. **Baseline.** Class-weighted logistic regression on the identical features. Is the deep model earning its complexity?
3. **Walk-forward (expanding-window) backtest.** This simulates monthly retraining in production. For each of 5 consecutive future blocks: refit the preprocessor and model on *all prior* wafers, with the last 20 % of that history as validation, then score the block. Pooled predictions cover far more defects than one test window.

In [12]:
seed_rows = []
for s in range(5):
    m_s, _ = train_sae(X_tr, M_tr, y_tr, X_va, y_va, seed=s, verbose=False)
    thr_s = pick_threshold(y_va, predict_proba(m_s, X_va))
    p_s = predict_proba(m_s, X_te)
    seed_rows.append(dict(seed=s, ap=average_precision_score(y_te, p_s), roc_auc=roc_auc_score(y_te, p_s),
                          f1=f1_score(y_te, p_s >= thr_s, zero_division=0)))
seed_df = pd.DataFrame(seed_rows)
print(seed_df.round(4).to_string(index=False))
print("mean ± std:", {k: f"{seed_df[k].mean():.4f} ± {seed_df[k].std():.4f}" for k in ["ap", "roc_auc", "f1"]})

lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000).fit(X_tr, y_tr)
thr_lr = pick_threshold(y_va, lr.predict_proba(X_va)[:, 1])
res_lr = evaluate(y_te, lr.predict_proba(X_te)[:, 1], thr_lr)
show(res_lr, "Logistic-regression baseline - same test window")

 seed     ap  roc_auc     f1
    0 0.0882   0.5811 0.0000
    1 0.0590   0.4977 0.0000
    2 0.0998   0.6498 0.0000
    3 0.0774   0.5918 0.0000
    4 0.0986   0.6300 0.1579
mean ± std: {'ap': '0.0846 ± 0.0169', 'roc_auc': '0.5901 ± 0.0587', 'f1': '0.0316 ± 0.0706'}


--- Logistic-regression baseline - same test window ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.0890  95% CI [0.048, 0.175]   (random = 0.0541)
ROC-AUC     0.6286  95% CI [0.482, 0.767]
F1          0.1412  95% CI [0.075, 0.212]   precision 0.0784  recall 0.7059
confusion   TN=156  FP=141  FN=5  TP=12
at natural prevalence 6.64%: precision 0.0956  F1 0.1684


In [13]:
def walk_forward(n_folds=5, start_frac=0.5):
    edges = np.linspace(int(start_frac * n), n, n_folds + 1).astype(int)
    ys, ps, preds, rows = [], [], [], []
    for k in range(n_folds):
        hist_ix, blk = np.arange(0, edges[k]), np.arange(edges[k], edges[k + 1])
        cut = int(0.8 * len(hist_ix)); tr_ix, va_ix = hist_ix[:cut], hist_ix[cut:]
        pp = CausalPreprocessor().fit(df.iloc[tr_ix])                     # refit on the past only
        (a, am), (b, _), (c, _) = (pp.transform(df.iloc[ix]) for ix in (tr_ix, va_ix, blk))
        m_k, _ = train_sae(a, am, y_all[tr_ix], b, y_all[va_ix], verbose=False)
        thr_k = pick_threshold(y_all[va_ix], predict_proba(m_k, b))
        p_k = predict_proba(m_k, c)
        yk = y_all[blk]
        ys.append(yk); ps.append(p_k); preds.append(p_k >= thr_k)
        rows.append(dict(fold=k, block=f"{df.timestamp[blk[0]]:%m-%d} -> {df.timestamp[blk[-1]]:%m-%d}",
                         n=len(blk), defects=int(yk.sum()),
                         ap=average_precision_score(yk, p_k) if yk.sum() else np.nan,
                         f1=f1_score(yk, p_k >= thr_k, zero_division=0)))
    y, p, pr = np.concatenate(ys), np.concatenate(ps), np.concatenate(preds)
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
    pooled = dict(n=len(y), defects=int(y.sum()), prevalence=y.mean(),
                  ap=average_precision_score(y, p), roc_auc=roc_auc_score(y, p),
                  f1=f1_score(y, pr), precision=precision_score(y, pr, zero_division=0),
                  recall=recall_score(y, pr), tp=tp, fp=fp, fn=fn, tn=tn)
    print(f"pooled walk-forward: n={pooled['n']} defects={pooled['defects']} prev={pooled['prevalence']:.2%} | "
          f"AP {pooled['ap']:.4f} | ROC-AUC {pooled['roc_auc']:.4f} | F1 {pooled['f1']:.4f} | "
          f"P {pooled['precision']:.4f} R {pooled['recall']:.4f} | TP={tp} FP={fp} FN={fn} TN={tn}")
    return pooled

res_wf = walk_forward()

 fold          block   n  defects     ap     f1
    0 09-11 -> 09-20 156        9 0.0883 0.1791
    1 09-20 -> 09-26 157        2 0.0203 0.0000
    2 09-26 -> 10-02 157        9 0.0531 0.0000
    3 10-02 -> 10-07 157       14 0.1027 0.1026
    4 10-07 -> 10-17 157        3 0.0436 0.0548
pooled walk-forward: n=784 defects=37 prev=4.72% | AP 0.0586 | ROC-AUC 0.5807 | F1 0.0816 | P 0.0481 R 0.2703 | TP=10 FP=198 FN=27 TN=549


# Stage B — pre-registered, leakage-safe tuning

Stage A showed two problems. The network **overfits**: train BCE falls to about 0.002 while validation AP stays flat. And the threshold picked from 11 validation defects is **unstable**: in 4 of 5 seeds F1 was 0.

**Protocol (fixed before any Stage B result was seen):**
1. The *development history* is train + val, the wafers before the test window. **The test window is not used in sections 12–13.**
2. Inside the history, a 4-fold expanding-window walk-forward CV. Each fold refits the preprocessor, the feature selector and the model on *past* wafers only, then scores the next block.
3. Every grid configuration is ranked by **pooled CV PR-AUC**. The winner, whichever model family it comes from, is the selected model.
4. The decision threshold is taken from the winner's **pooled CV predictions**, which hold more defects than the 11-defect validation window.
5. The winner is refit on the full history (preprocessor, selector, 5-seed ensemble) and scored **once** on the test window. That result is what the release gate sees.

**Grid.**
* SAE (5-seed ensemble during final fitting, 3 seeds in CV): hidden {64, 128} × latent {8, 16} × weight-decay {1e-3, 1e-2} × epochs {10, 25} × features {all, top-60}. Dropout is 0.3 throughout.
* Class-weighted logistic regression: C {1e-3, 1e-2, 1e-1} × features {all, top-60}.
* *Top-60*: the 60 features with the highest |point-biserial correlation| with the label, computed on the fold's training rows only, plus the 5 legacy/categorical features, which are always kept.

## 12 · Walk-forward CV inside the development history

In [14]:
def select_features(X, y, k):
    # train-only univariate screen; the legacy/categorical block (last 5) is always kept
    if k is None:
        return np.arange(X.shape[1])
    body = X[:, :-5]
    yc = y - y.mean()
    xc = body - body.mean(0)
    denom = np.sqrt((xc ** 2).sum(0) * (yc ** 2).sum())
    corr = np.abs(np.divide(xc.T @ yc, denom, out=np.zeros(body.shape[1]), where=denom > 0))
    top = np.sort(np.argsort(-corr)[:k])
    return np.concatenate([top, np.arange(X.shape[1] - 5, X.shape[1])])

def train_sae_fixed(X, M, y, cfg, seed):
    # fixed-epoch training: no peeking at any later window for early stopping
    seed_all(seed)
    model = WaferSAE(X.shape[1], cfg["hidden"], cfg["latent"], p_drop=cfg["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    pos_weight = torch.tensor((1 - y).sum() / max(y.sum(), 1), dtype=torch.float32)
    loader = make_loader(X, M, y, shuffle=True, seed=seed)
    for _ in range(cfg["epochs"]):
        model.train()
        for xb, mb, yb in loader:
            r, logit, _ = model(xb)
            loss, _, _ = sae_loss(r, xb, mb, logit, yb, pos_weight, HP["lambda_clf"])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); opt.step()
        sched.step()
    return model

class FittedPipeline:
    # preprocessor + selector + model(s), all fit on one block of past wafers
    def __init__(self, cfg, seeds):
        self.cfg, self.seeds = cfg, seeds

    def fit(self, frame, y):
        self.prep = CausalPreprocessor().fit(frame)
        X, M = self.prep.transform(frame)
        self.cols = select_features(X, y, self.cfg["k"])
        X, M = X[:, self.cols], M[:, self.cols]
        if self.cfg.get("smote"):
            # SMOTE on THIS block's training rows only, after train-only preprocessing (never touches eval rows)
            n0 = len(X)
            X, y = SMOTE(random_state=SEED, k_neighbors=min(5, int(y.sum()) - 1)).fit_resample(X, y)
            X = X.astype(np.float32)
            M = np.concatenate([M, np.ones((len(X) - n0, M.shape[1]), np.float32)])
        if self.cfg["kind"] == "lr":
            self.models = [LogisticRegression(C=self.cfg["C"],
                                              class_weight=None if self.cfg.get("smote") else "balanced",
                                              max_iter=5000).fit(X, y)]
        else:
            self.models = [train_sae_fixed(X, M, y, self.cfg, s) for s in self.seeds]
        return self

    def member_proba(self, frame):
        X = self.prep.transform(frame)[0][:, self.cols]
        if self.cfg["kind"] == "lr":
            return [self.models[0].predict_proba(X)[:, 1]]
        return [predict_proba(m, X) for m in self.models]

    def predict_proba(self, frame):
        return np.mean(self.member_proba(frame), axis=0)

GRID = ([dict(kind="sae", hidden=h, latent=l, dropout=0.3, wd=wd, epochs=e, k=k)
         for h in (64, 128) for l in (8, 16) for wd in (1e-3, 1e-2) for e in (10, 25) for k in (None, 60)]
        + [dict(kind="lr", C=C, k=k) for C in (1e-3, 1e-2, 1e-1) for k in (None, 60)])

dev_ix = np.arange(0, i2)                                  # train + val history; test window excluded
cv_edges = np.linspace(len(dev_ix) // 2, len(dev_ix), 5).astype(int)
for k in range(4):
    blk = np.arange(cv_edges[k], cv_edges[k + 1])
    print(f"CV fold {k}: fit on wafers [0, {cv_edges[k]}) -> score [{cv_edges[k]}, {cv_edges[k+1]}) "
          f"defects={y_all[blk].sum()}")

def cv_predict(cfg, seeds=(0, 1, 2)):
    ys, ps = [], []
    for k in range(4):
        past, blk = np.arange(0, cv_edges[k]), np.arange(cv_edges[k], cv_edges[k + 1])
        pipe = FittedPipeline(cfg, seeds).fit(df.iloc[past], y_all[past])
        ys.append(y_all[blk]); ps.append(pipe.predict_proba(df.iloc[blk]))
    return np.concatenate(ys), np.concatenate(ps)

cv_rows, cv_preds = [], {}
for gi, cfg in enumerate(GRID):
    yy, pp = cv_predict(cfg)
    cv_preds[gi] = (yy, pp)
    cv_rows.append(dict(id=gi, **{k: cfg.get(k) for k in ["kind", "hidden", "latent", "wd", "epochs", "C", "k"]},
                        cv_ap=average_precision_score(yy, pp), cv_auc=roc_auc_score(yy, pp)))
cv_df = pd.DataFrame(cv_rows).sort_values("cv_ap", ascending=False)
print(f"pooled CV: n={len(yy)} defects={yy.sum()} prevalence={yy.mean():.2%}")
print(cv_df.head(12).round(4).to_string(index=False))
print("\nbest per family:")
print(cv_df.groupby("kind").head(1).round(4).to_string(index=False))

CV fold 0: fit on wafers [0, 626) -> score [626, 782) defects=2
CV fold 1: fit on wafers [0, 782) -> score [782, 939) defects=9
CV fold 2: fit on wafers [0, 939) -> score [939, 1096) defects=2
CV fold 3: fit on wafers [0, 1096) -> score [1096, 1253) defects=9


pooled CV: n=627 defects=22 prevalence=3.51%
 id kind  hidden  latent    wd  epochs   C    k  cv_ap  cv_auc
 37   lr     NaN     NaN   NaN     NaN 0.1 60.0 0.1163  0.5669
 15  sae    64.0    16.0 0.010    25.0 NaN 60.0 0.0730  0.5482
 11  sae    64.0    16.0 0.001    25.0 NaN 60.0 0.0730  0.5481
 27  sae   128.0    16.0 0.001    25.0 NaN 60.0 0.0726  0.5789
 31  sae   128.0    16.0 0.010    25.0 NaN 60.0 0.0726  0.5788
 10  sae    64.0    16.0 0.001    25.0 NaN  NaN 0.0672  0.4842
 14  sae    64.0    16.0 0.010    25.0 NaN  NaN 0.0672  0.4841
 22  sae   128.0     8.0 0.010    25.0 NaN  NaN 0.0661  0.4539
 18  sae   128.0     8.0 0.001    25.0 NaN  NaN 0.0661  0.4539
 23  sae   128.0     8.0 0.010    25.0 NaN 60.0 0.0651  0.5515
 19  sae   128.0     8.0 0.001    25.0 NaN 60.0 0.0651  0.5514
  7  sae    64.0     8.0 0.010    25.0 NaN 60.0 0.0647  0.5655

best per family:
 id kind  hidden  latent   wd  epochs   C    k  cv_ap  cv_auc
 37   lr     NaN     NaN  NaN     NaN 0.1 60.0 0.1163  0

## 13 · Selected model: refit on full history, threshold from CV, one test evaluation

In [15]:
best_id = int(cv_df.iloc[0]["id"])
BEST_CFG = GRID[best_id]
yy, pp = cv_preds[best_id]
THRESHOLD_B = pick_threshold(yy, pp)
cv_pred = pp >= THRESHOLD_B
print(f"selected config #{best_id}: {BEST_CFG}")
print(f"CV pooled: AP {average_precision_score(yy, pp):.4f} | F1@thr {f1_score(yy, cv_pred):.4f} "
      f"| threshold {THRESHOLD_B:.4f}")

final_pipe = FittedPipeline(BEST_CFG, seeds=range(5)).fit(df.iloc[dev_ix], y_all[dev_ix])
members = final_pipe.member_proba(df.iloc[idx_te])
p_te_B = np.mean(members, axis=0)
res_B = evaluate(y_te, p_te_B, THRESHOLD_B)
show(res_B, f"Stage B selected model ({BEST_CFG['kind']}) - chronological test window, scored once")
print(classification_report(y_te, (p_te_B >= THRESHOLD_B).astype(int), target_names=["pass", "defect"], digits=4))
if len(members) > 1:
    print("individual ensemble members on test (AP / F1@thr):",
          [(round(average_precision_score(y_te, m), 4), round(f1_score(y_te, m >= THRESHOLD_B, zero_division=0), 4))
           for m in members])

selected config #37: {'kind': 'lr', 'C': 0.1, 'k': 60}
CV pooled: AP 0.1163 | F1@thr 0.1935 | threshold 0.6240


--- Stage B selected model (lr) - chronological test window, scored once ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.2189  95% CI [0.057, 0.432]   (random = 0.0541)
ROC-AUC     0.5558  95% CI [0.402, 0.719]
F1          0.1905  95% CI [0.047, 0.348]   precision 0.1600  recall 0.2353
confusion   TN=276  FP=21  FN=13  TP=4
at natural prevalence 6.64%: precision 0.1914  F1 0.2111
              precision    recall  f1-score   support

        pass     0.9550    0.9293    0.9420       297
      defect     0.1600    0.2353    0.1905        17

    accuracy                         0.8917       314
   macro avg     0.5575    0.5823    0.5662       314
weighted avg     0.9120    0.8917    0.9013       314



### 14 · Reference only: best configuration of the *other* family
This is shown for transparency. It was **not** used for selection, because selection was already fixed by CV in section 12.

In [16]:
other = cv_df[cv_df["kind"] != BEST_CFG["kind"]].iloc[0]
ocfg = GRID[int(other["id"])]
oy, op = cv_preds[int(other["id"])]
othr = pick_threshold(oy, op)
opipe = FittedPipeline(ocfg, seeds=range(5)).fit(df.iloc[dev_ix], y_all[dev_ix])
res_other = evaluate(y_te, opipe.predict_proba(df.iloc[idx_te]), othr)
show(res_other, f"reference: best {ocfg['kind']} config {ocfg}")

--- reference: best sae config {'kind': 'sae', 'hidden': 64, 'latent': 16, 'dropout': 0.3, 'wd': 0.01, 'epochs': 25, 'k': 60} ---
n=314  defects=17  prevalence=5.41%
PR-AUC (AP) 0.1292  95% CI [0.057, 0.326]   (random = 0.0541)
ROC-AUC     0.6201  95% CI [0.480, 0.770]
F1          0.2632  95% CI [0.074, 0.444]   precision 0.2381  recall 0.2941
confusion   TN=281  FP=16  FN=12  TP=5
at natural prevalence 6.64%: precision 0.2797  F1 0.2867


### 14b · Sensitivity: grid v2 = v1 + SMOTE-on-train variants
**Disclosure:** these configurations were added *after* the v1 test result above had been seen, because a reviewer asked whether SMOTE applied **to training rows only** helps.
Selection still uses only the CV inside the development history. The v1 pre-registered result stays the **primary** result. This section is reported as a sensitivity check.
SMOTE runs inside each fold, after that fold's train-only preprocessing and feature selection, so synthetic wafers never reach a scoring block. For SMOTE variants the class weighting is turned off (`pos_weight` ≈ 1, `class_weight=None`) to avoid double-counting.

In [17]:
GRID_V2 = [dict(c, smote=True) for c in GRID]
offset = len(GRID)
for gi, cfg in enumerate(GRID_V2):
    yy2, pp2 = cv_predict(cfg)
    cv_preds[offset + gi] = (yy2, pp2)
    cv_rows.append(dict(id=offset + gi, **{k: cfg.get(k) for k in ["kind", "hidden", "latent", "wd", "epochs", "C", "k"]},
                        smote=True, cv_ap=average_precision_score(yy2, pp2), cv_auc=roc_auc_score(yy2, pp2)))
ALL_GRID = GRID + GRID_V2
cv_df2 = pd.DataFrame(cv_rows).fillna({"smote": False}).sort_values("cv_ap", ascending=False)
print(cv_df2.head(10).round(4).to_string(index=False))
print("\nbest per (family, smote):")
print(cv_df2.groupby(["kind", "smote"]).head(1).round(4).to_string(index=False))

best2 = int(cv_df2.iloc[0]["id"])
CFG_V2 = ALL_GRID[best2]
if best2 == best_id:
    print(f"\nv2 CV selects the same configuration as v1 (#{best_id}); test result unchanged.")
    res_v2, thr_v2 = res_B, THRESHOLD_B
else:
    yy2, pp2 = cv_preds[best2]
    thr_v2 = pick_threshold(yy2, pp2)
    pipe_v2 = FittedPipeline(CFG_V2, seeds=range(5)).fit(df.iloc[dev_ix], y_all[dev_ix])
    p_v2 = pipe_v2.predict_proba(df.iloc[idx_te])
    res_v2 = evaluate(y_te, p_v2, thr_v2)
    show(res_v2, f"grid v2 CV-selected config {CFG_V2}")
    print(classification_report(y_te, (p_v2 >= thr_v2).astype(int), target_names=["pass", "defect"], digits=4))

 id kind  hidden  latent    wd  epochs   C    k  cv_ap  cv_auc  smote
 37   lr     NaN     NaN   NaN     NaN 0.1 60.0 0.1163  0.5669  False
 75   lr     NaN     NaN   NaN     NaN 0.1 60.0 0.1113  0.5513   True
 60  sae   128.0     8.0 0.010    25.0 NaN  NaN 0.0906  0.4660   True
 56  sae   128.0     8.0 0.001    25.0 NaN  NaN 0.0906  0.4654   True
 15  sae    64.0    16.0 0.010    25.0 NaN 60.0 0.0730  0.5482  False
 11  sae    64.0    16.0 0.001    25.0 NaN 60.0 0.0730  0.5481  False
 27  sae   128.0    16.0 0.001    25.0 NaN 60.0 0.0726  0.5789  False
 31  sae   128.0    16.0 0.010    25.0 NaN 60.0 0.0726  0.5788  False
 10  sae    64.0    16.0 0.001    25.0 NaN  NaN 0.0672  0.4842  False
 14  sae    64.0    16.0 0.010    25.0 NaN  NaN 0.0672  0.4841  False

best per (family, smote):
 id kind  hidden  latent   wd  epochs   C    k  cv_ap  cv_auc  smote
 37   lr     NaN     NaN  NaN     NaN 0.1 60.0 0.1163  0.5669  False
 75   lr     NaN     NaN  NaN     NaN 0.1 60.0 0.1113  0.5513   T

C:\Users\Arav Kilak\AppData\Local\Temp\ipykernel_54464\3343092015.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cv_df2 = pd.DataFrame(cv_rows).fillna({"smote": False}).sort_values("cv_ap", ascending=False)


## 15 · Release gate (honest; uses the Stage B pre-registered selected model)

In [18]:
GATE_F1 = 0.90
passed = res_B["f1"] >= GATE_F1
print(f"Release gate (F1 ≥ {GATE_F1}) on natural-prevalence test window: {res_B['f1']:.4f} -> "
      f"{'PASSED' if passed else 'FAILED - NO-GO'}")

def _js(r):
    return dict((k, float(v) if np.isscalar(v) else [float(x) for x in v]) for k, v in r.items())

os.makedirs("artifacts", exist_ok=True)
joblib.dump(dict(pipeline=final_pipe, threshold=THRESHOLD_B), "artifacts/selected_pipeline.joblib")
summary = dict(gate_passed=bool(passed),
               stage_b=dict(config=BEST_CFG, threshold=THRESHOLD_B, test=_js(res_B),
                            reference_other_family=dict(config=ocfg, test=_js(res_other)),
                            cv_table=cv_df2.to_dict(orient="records"),
                            sensitivity_v2=dict(config=CFG_V2, threshold=thr_v2, test=_js(res_v2))),
               stage_a=dict(threshold=THRESHOLD, test=_js(res_sae), baseline_logreg=_js(res_lr),
                            walk_forward=_js(res_wf), seeds=seed_df.to_dict(orient="records")))
with open("artifacts/metrics.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("artifacts written: selected_pipeline.joblib, metrics.json "
      "(for traceability only - the gate decision is NO-GO unless PASSED above)")

Release gate (F1 ≥ 0.9) on natural-prevalence test window: 0.1905 -> FAILED - NO-GO
artifacts written: selected_pipeline.joblib, metrics.json (for traceability only - the gate decision is NO-GO unless PASSED above)
